# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [84]:
class Matrix:
    # Question 1
    def __init__(self, *args):
        if len(args) == 2 and isinstance(args[0], int) and isinstance(args[1], int):
            n, m = args
            if n <= 0 or m <= 0:
              raise ValueError("Matrix Dimensions must be positive values.")
            self.n = n
            self.m = m
            self.matrix = [[0 for _ in range(m)] for _ in range(n)]
        elif len(args) == 1 and isinstance(args[0], list):
            list_of_lists = args[0]
            if not list_of_lists:
               raise ValueError("Cannot initialize an empty matrix from an empty list.")
            self.n = len(list_of_lists)
            if not all(isinstance(row, list) for row in list_of_lists):
               raise ValueError("All elements in the list must be lists(rows).")
            if not all(len(row) > 0 for row in list_of_lists) and self.n > 0:
               raise ValueError("All rows must contain at least one element.")
            self.m = len(list_of_lists[0]) if self.n > 0 else 0
            for i, row in enumerate(list_of_lists):
              if len(row) != self.m:
                 raise ValueError(f"Row {i} has {len(row)} columns, but expected {self.m} for a rectangular matrix.")
            self.matrix = [row[:] for row in list_of_lists]
        else:
           raise TypeError("Matrix must be intialized with either (n, m) or a list of lists")
            
    def __getitem__(self, index):
        if isinstance(index, tuple):
          if len(index) != 2:
            raise TypeError("Tuple index must have at least two elements")
          i, j = index
          if isinstance(i, int) and isinstance(j, int):
            rows = self.matrix[i]
            if isinstance(j, int):
              result = [row[j] for row in rows]
            else:
              result = [row[j[0]:j[1]] for row in rows]
            return Matrix(result)
          return self.matrix[i][j]
        elif isinstance(index, int):
          return self.matrix[index]

        else:
          raise TypeError("Matrix index must be an integer or a tuple")

    def __setitem__(self, index, value):
        if isinstance(index, tuple):
            if len(index) != 2:
                raise TypeError("Tuple index must have two elements")
            i, j = index
            if isinstance(i, int) and isinstance(j, int):
                self.matrix[i][j] = value
        elif isinstance(index, int):
            self.matrix[index] = value
        else:
            raise TypeError("Matrix index must be an integer or a tuple")
    def __eq__(self, other):
        if self.n != other.n or self.m != other.m:
            return False
        for i in range(self.n):
            for j in range(self.m):
                if self.matrix[i][j] != other.matrix[i][j]:
                    return False
        return True
    def __str__(self):
        return str(self.matrix)

    # Question 2
    def shape(self):
        return (self.n, self.m)

    def transpose(self):
        matrix = [[0 for j in range(self.n)] for i in range(self.m)]
        for i in range(self.n):
            for j in range(self.m):
                matrix[j][i] = self.matrix[i][j]
        return Matrix(matrix)

    def row(self, n):
        return Matrix([[self.matrix[n][j] for j in range(self.m)]])
    def column(self, n):
        col_list = [[self.matrix[i][n]] for i in range(self.n)]
        return Matrix(col_list)

    def to_list(self):
        return [row[:] for row in self.matrix]
    def block(self, n_0, n_1, m_0, m_1):
        if not (0 <= m_0 <= m_1 < self.n and 0 <= n_0 <= n_1 < self.m):
            raise IndexError("Index out of range")
        num_rows = m_1 - m_0 + 1
        num_cols = n_1 - n_0 + 1
        block_matrix = [[0 for _ in range(num_cols)] for _ in range(num_rows)]
        for i in range(num_rows):
            for j in range(num_cols):
                block_matrix[i][j] = self.matrix[m_0 + i][n_0 + j]
        return Matrix(block_matrix)

    # Question 4
    def scalarmul(self, c):
        return Matrix([[c * self.matrix[i][j] for j in range(self.m)] for i in range(self.n)])

    def add(self, other):
        if self.n != other.n or self.m != other.m:
            raise ValueError("Matrix must have the same dimensions")
        return Matrix([[self.matrix[i][j] + other.matrix[i][j] for j in range(self.m)] for i in range(self.n)])

    def sub(self, other):
        if self.n != other.n or self.m != other.m:
            raise ValueError("Matrix must have the same dimensions")
        return Matrix([[self.matrix[i][j] - other.matrix[i][j] for j in range(self.m)] for i in range(self.n)])

    def mat_mult(self, other):
        if self.m != other.n:
            raise ValueError("Matrix can't work with multiplcation")
        result = [[0 for _ in range(other.m)] for _ in range(self.n)]
        for i in range(self.n):
            for j in range(other.m):
                for k in range(self.m):
                    result[i][j] += self.matrix[i][k] * other.matrix[k][j]
        return Matrix(result)

    def element_mult(self, other):
        if self.n != other.n or self.m != other.m:
            raise ValueError("Matrix must have same dimensions")
        return Matrix([[self.matrix[i][j] * other.matrix[i][j] for j in range(self.m)] for i in range(self.n)])

    # Question 5    
    def __add__(self, other):
        return self.add(other)
    def __sub__(self, other):
        return self.sub(other)

    def equals(self, other):
        if self.n != other.n or self.m != other.m:
            return False
        for i in range(self.n):
            for j in range(self.m):
                if self.matrix[i][j] != other.matrix[i][j]:
                    return False
        return True

    def __mul__(self, other):
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        elif isinstance(other, Matrix):
            return self.mat_mult(other)
        return NotImplemented
    def __rmul__(self, other):
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        return NotImplemented

In [85]:
# Question 3
def constant(n, m, c):
    return Matrix([[float(c) for _ in range(m)] for _ in range(n)])

def zeros(n, m):
    return Matrix([[0 for _ in range(m)] for _ in range(n)])

def ones(n, m):
    return Matrix([[1 for _ in range(m)] for _ in range(n)])

def eye(n):
    return Matrix([[1 if i == j else 0 for j in range(n)] for i in range(n)])

In [87]:
# Question 6
A = Matrix([[1,2], [3,4]])
B = Matrix([[5,6],[7,8]])
C = Matrix([[2,4],[6,9]])
I = eye(2)
print("Problem 1")
matrics1 = (A * B) * C
print(f"((AB)C: {matrics1}")

matrics2 = A * (B * C)
print(f"((BC)A: {matrics2}")

print("Problem 2")
matrics3 = A * (B + C)
matrics4 = A*B + A*C
print(f"A * (B + C): {matrics3}")
print(f"AB + AC: {matrics4}")
print("Problem 3")
matrics5 = A*B
matrics6 = B*A
print(f"A*B: {matrics5}")
print(f"B*A: {matrics6}")
print("AB does not equal to BA")
print("Problem 4")
matrics7 = A * I
matrics8 = A
print(f"A * I: {matrics7}")
print(f"A: {matrics8}")

Problem 1
((AB)C: [[170, 274], [386, 622]]
((BC)A: [[170, 274], [386, 622]]
Problem 2
A * (B + C): [[33, 44], [73, 98]]
AB + AC: [[33, 44], [73, 98]]
Problem 3
A*B: [[19, 22], [43, 50]]
B*A: [[23, 34], [31, 46]]
AB does not equal to BA
Problem 4
A * I: [[1, 2], [3, 4]]
A: [[1, 2], [3, 4]]


In [88]:
# In this lab I used LLMs to check to see if i made any mistakes on here. I've made some mistakes with the __getitem__ variable.
# I am unsure if I was able to modify it correctly.
# While working on this lab, I made an error.
# I had to copy and paste it on colab to see where the problem was. On here I had trouble trying to locate where the problem was.
# Overall the matrix was able to function well on here.